In [0]:
USE CATALOG dbw_kroger_insights;

In [0]:
describe gold.dim_date;

In [0]:
WITH daily AS (
    SELECT
    d.date,
    d.day_name,
    d.is_weekend,
    d.is_black_friday,
    COUNT(s.user_session)                                                   AS sessions,
    ROUND(100.0 * COUNT_IF(s.has_purchase) / NULLIF(COUNT(s.user_session), 0), 2) AS session_conversion_pct,
    ROUND(COALESCE(SUM(s.revenue), 0), 2)                                   AS revenue,
    ROUND(COALESCE(SUM(s.revenue), 0) / NULLIF(COUNT(s.user_session), 0), 2) AS revenue_per_session
FROM dbw_kroger_insights.gold.dim_date d
LEFT JOIN dbw_kroger_insights.gold.ec_fact_sessions s
    ON s.session_date = d.date
GROUP BY d.date, d.day_name, d.is_weekend, d.is_black_friday
)
SELECT COUNT(*) AS days, SUM(sessions) AS sessions, ROUND(SUM(revenue), 2) AS revenue
FROM daily;

In [0]:
SELECT 
    COALESCE(d.date, s.session_date)                    AS date,
    d.day_name,
    d.is_weekend,
    d.is_black_friday,
    COUNT(s.user_session)                               AS sessions,
    ROUND(100.0 * COUNT_IF(s.has_purchase) / NULLIF(COUNT(*), 0), 2) AS session_conversion_pct,
    ROUND(SUM(s.revenue), 2)                            AS revenue,
    ROUND(SUM(s.revenue) / NULLIF(COUNT(*), 0), 2)      AS revenue_per_session
FROM dbw_kroger_insights.gold.ec_fact_sessions s
LEFT JOIN dbw_kroger_insights.gold.dim_date d 
    ON s.session_date = d.date
WHERE d.date BETWEEN DATE'2019-11-25' AND DATE'2019-11-30'
GROUP BY 
    COALESCE(d.date, s.session_date),
    d.day_name,
    d.is_weekend,
    d.is_black_friday
ORDER BY 
    date ASC;

In [0]:
WITH daily_rollup AS (
    SELECT
        d.date,
        d.is_weekend,
        COUNT(s.user_session)                  AS sessions,
        COUNT_IF(s.has_purchase)               AS converting_sessions,
        COALESCE(SUM(s.revenue), 0)            AS revenue
    FROM dbw_kroger_insights.gold.dim_date d
    LEFT JOIN dbw_kroger_insights.gold.ec_fact_sessions s
        ON s.session_date = d.date
    GROUP BY d.date, d.is_weekend
)
SELECT
    CASE WHEN is_weekend THEN 'Weekend' ELSE 'Weekday' END             AS day_type,
    COUNT(*)                                                          AS days,
    ROUND(AVG(sessions), 0)                                           AS avg_daily_sessions,
    ROUND(100.0 * SUM(converting_sessions) / NULLIF(SUM(sessions), 0), 2) AS conversion_pct,
    ROUND(SUM(revenue) / NULLIF(SUM(sessions), 0), 2)                 AS revenue_per_session,
    ROUND(AVG(revenue), 0)                                            AS avg_daily_revenue
FROM daily_rollup
GROUP BY is_weekend
ORDER BY is_weekend;

In [0]:
SELECT 
    d.date,
    d.day_name,
    d.is_weekend,
    d.is_black_friday,
    ROUND(SUM(s.revenue), 2) AS total_revenue
FROM dbw_kroger_insights.gold.dim_date d
LEFT JOIN dbw_kroger_insights.gold.ec_fact_sessions s
    ON s.session_date = d.date
GROUP BY 
    d.date,
    d.day_name,
    d.is_weekend,
    d.is_black_friday
ORDER BY 
    total_revenue DESC
LIMIT 5;

In [0]:
SELECT 
    d.date,
    d.day_name,
    d.is_weekend,
    d.is_black_friday,
    COUNT(s.user_session)                               AS sessions,
    ROUND(100.0 * COUNT_IF(s.has_purchase) / NULLIF(COUNT(s.user_session), 0), 2) AS session_conversion_pct,
    ROUND(SUM(s.revenue), 2)                            AS revenue,
    ROUND(SUM(s.revenue) / NULLIF(COUNT(s.user_session), 0), 2) AS revenue_per_session,
    ROUND(SUM(s.revenue) / NULLIF(COUNT_IF(s.has_purchase), 0), 2) AS revenue_per_converting_session,
    ROUND(SUM(s.purchases) * 1.0 / NULLIF(COUNT_IF(s.has_purchase), 0), 2) AS items_per_converting_session
FROM dbw_kroger_insights.gold.dim_date d
LEFT JOIN dbw_kroger_insights.gold.ec_fact_sessions s
    ON s.session_date = d.date
WHERE d.date BETWEEN DATE'2019-11-14' AND DATE'2019-11-19'
GROUP BY 
    d.date,
    d.day_name,
    d.is_weekend,
    d.is_black_friday
ORDER BY 
    d.date ASC;

In [0]:
WITH daily_rollup AS (
    SELECT
        d.date,
        d.is_weekend,
        COUNT(s.user_session)                               AS sessions,
        COUNT_IF(s.has_purchase)                            AS converting_sessions,
        COALESCE(SUM(s.revenue), 0)                         AS revenue
    FROM dbw_kroger_insights.gold.dim_date d
    LEFT JOIN dbw_kroger_insights.gold.ec_fact_sessions s
        ON s.session_date = d.date
    GROUP BY d.date, d.is_weekend
)
SELECT
    CASE WHEN is_weekend THEN 'Weekend' ELSE 'Weekday' END             AS day_type,
    COUNT(*)                                                          AS days,
    ROUND(AVG(sessions), 0)                                           AS avg_daily_sessions,
    ROUND(percentile_approx(revenue, 0.5), 2)                         AS median_daily_revenue,
    ROUND(AVG(revenue), 0)                                            AS avg_daily_revenue,
    ROUND(100.0 * SUM(converting_sessions) / NULLIF(SUM(sessions), 0), 2) AS conversion_pct,
    ROUND(SUM(revenue) / NULLIF(SUM(sessions), 0), 2)                 AS revenue_per_session
FROM daily_rollup
WHERE date NOT IN (DATE'2019-11-16', DATE'2019-11-17')
GROUP BY is_weekend
ORDER BY is_weekend;

In [0]:
WITH daily_rollup AS (
    SELECT
        d.date,
        d.is_weekend,
        COUNT(s.user_session)                               AS sessions,
        COUNT_IF(s.has_purchase)                            AS converting_sessions,
        COALESCE(SUM(s.revenue), 0)                         AS revenue
    FROM dbw_kroger_insights.gold.dim_date d
    LEFT JOIN dbw_kroger_insights.gold.ec_fact_sessions s
        ON s.session_date = d.date
    GROUP BY d.date, d.is_weekend
)
SELECT
    CASE WHEN is_weekend THEN 'Weekend' ELSE 'Weekday' END             AS day_type,
    COUNT(*)                                                          AS days,
    ROUND(AVG(sessions), 0)                                           AS avg_daily_sessions,
    ROUND(percentile_approx(revenue, 0.5), 2)                         AS median_daily_revenue,
    ROUND(AVG(revenue), 0)                                            AS avg_daily_revenue,
    ROUND(100.0 * SUM(converting_sessions) / NULLIF(SUM(sessions), 0), 2) AS conversion_pct,
    ROUND(SUM(revenue) / NULLIF(SUM(sessions), 0), 2)                 AS revenue_per_session
FROM daily_rollup
WHERE date NOT BETWEEN DATE'2019-11-14' AND DATE'2019-11-17'
GROUP BY is_weekend
ORDER BY is_weekend;

In [0]:
WITH daily_rollup AS (
    SELECT
        d.date,
        d.month_num,
        d.month_name,
        COUNT(s.user_session)                               AS sessions,
        COUNT_IF(s.has_purchase)                            AS converting_sessions,
        COALESCE(SUM(s.revenue), 0)                         AS revenue
    FROM dbw_kroger_insights.gold.dim_date d
    LEFT JOIN dbw_kroger_insights.gold.ec_fact_sessions s
        ON s.session_date = d.date
    WHERE d.month_num IN (10, 11)
    GROUP BY d.date, d.month_num, d.month_name
)
SELECT
    month_name,
    COUNT(*)                                                              AS days,
    ROUND(AVG(sessions), 0)                                               AS avg_daily_sessions,
    ROUND(percentile_approx(revenue, 0.5), 2)                             AS median_daily_revenue,
    ROUND(100.0 * SUM(converting_sessions) / NULLIF(SUM(sessions), 0), 2) AS conversion_pct,
    ROUND(SUM(revenue) / NULLIF(SUM(sessions), 0), 2)                     AS revenue_per_session,
    ROUND(SUM(revenue), 2)                                                AS total_revenue
FROM daily_rollup
GROUP BY month_num, month_name
ORDER BY month_num;

In [0]:
WITH daily_rollup AS (
    SELECT
        d.date,
        d.month_num,
        d.month_name,
        COUNT(s.user_session)                               AS sessions,
        COUNT_IF(s.has_purchase)                            AS converting_sessions,
        COALESCE(SUM(s.revenue), 0)                         AS revenue
    FROM dbw_kroger_insights.gold.dim_date d
    LEFT JOIN dbw_kroger_insights.gold.ec_fact_sessions s
        ON s.session_date = d.date
    WHERE d.month_num IN (10, 11)
      AND NOT (d.date BETWEEN DATE'2019-11-14' AND DATE'2019-11-17')
    GROUP BY d.date, d.month_num, d.month_name
),
monthly_rollup AS (
    SELECT
        month_name,
        COUNT(*)                                                              AS days,
        ROUND(AVG(sessions), 0)                                               AS avg_daily_sessions,
        ROUND(percentile_approx(revenue, 0.5), 2)                             AS median_daily_revenue,
        ROUND(100.0 * SUM(converting_sessions) / NULLIF(SUM(sessions), 0), 2) AS conversion_pct,
        ROUND(SUM(revenue) / NULLIF(SUM(sessions), 0), 2)                     AS revenue_per_session,
        ROUND(SUM(revenue), 2)                                                AS total_revenue
    FROM daily_rollup
    GROUP BY month_num, month_name
)
SELECT
    -- Average Daily Sessions Comparison
    MAX(CASE WHEN month_name = 'October' THEN avg_daily_sessions END) AS oct_avg_daily_sessions,
    MAX(CASE WHEN month_name = 'November' THEN avg_daily_sessions END) AS nov_avg_daily_sessions,
    ROUND(
        (MAX(CASE WHEN month_name = 'November' THEN avg_daily_sessions END) - 
         MAX(CASE WHEN month_name = 'October' THEN avg_daily_sessions END)) * 100.0 / 
         NULLIF(MAX(CASE WHEN month_name = 'October' THEN avg_daily_sessions END), 0), 2
    ) AS avg_daily_sessions_pct_change,

    -- Conversion Rate Comparison (Percentage Points)
    MAX(CASE WHEN month_name = 'October' THEN conversion_pct END) AS oct_conversion_pct,
    MAX(CASE WHEN month_name = 'November' THEN conversion_pct END) AS nov_conversion_pct,
    ROUND(
        MAX(CASE WHEN month_name = 'November' THEN conversion_pct END) - 
        MAX(CASE WHEN month_name = 'October' THEN conversion_pct END), 2
    ) AS conversion_change_pp,

    -- Revenue Per Session Comparison
    MAX(CASE WHEN month_name = 'October' THEN revenue_per_session END) AS oct_revenue_per_session,
    MAX(CASE WHEN month_name = 'November' THEN revenue_per_session END) AS nov_revenue_per_session,
    ROUND(
        MAX(CASE WHEN month_name = 'November' THEN revenue_per_session END) - 
        MAX(CASE WHEN month_name = 'October' THEN revenue_per_session END), 2
    ) AS revenue_per_session_change,

    -- Total Revenue Comparison (%)
    MAX(CASE WHEN month_name = 'October' THEN total_revenue END) AS oct_total_revenue,
    MAX(CASE WHEN month_name = 'November' THEN total_revenue END) AS nov_total_revenue,
    ROUND(
        (MAX(CASE WHEN month_name = 'November' THEN total_revenue END) - 
         MAX(CASE WHEN month_name = 'October' THEN total_revenue END)) * 100.0 / 
         NULLIF(MAX(CASE WHEN month_name = 'October' THEN total_revenue END), 0), 2
    ) AS total_revenue_pct_change
FROM monthly_rollup;

In [0]:
WITH daily_rollup AS (
    SELECT
        d.date,
        d.day_name,
        d.is_weekend,
        d.is_black_friday,
        COUNT(s.user_session)                               AS sessions,
        COUNT_IF(s.has_purchase)                            AS converting_sessions,
        COALESCE(SUM(s.revenue), 0)                         AS revenue
    FROM dbw_kroger_insights.gold.dim_date d
    LEFT JOIN dbw_kroger_insights.gold.ec_fact_sessions s
        ON s.session_date = d.date
    WHERE d.date BETWEEN DATE'2019-10-01' AND DATE'2019-11-30'
    GROUP BY d.date, d.day_name, d.is_weekend, d.is_black_friday
)
SELECT
    date,
    day_name,
    sessions,
    revenue,
    ROUND(AVG(sessions) OVER (
        ORDER BY date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 0) AS sessions_7d_avg,
    ROUND(AVG(revenue) OVER (
        ORDER BY date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) AS revenue_7d_avg
FROM daily_rollup
ORDER BY date ASC;

In [0]:
WITH daily_rollup AS (
    SELECT
        d.date,
        d.day_name,
        d.is_weekend,
        d.is_black_friday,
        COUNT(s.user_session)                               AS sessions,
        COUNT_IF(s.has_purchase)                            AS converting_sessions,
        COALESCE(SUM(s.revenue), 0)                         AS revenue
    FROM dbw_kroger_insights.gold.dim_date d
    LEFT JOIN dbw_kroger_insights.gold.ec_fact_sessions s
        ON s.session_date = d.date
    WHERE d.date BETWEEN DATE'2019-10-01' AND DATE'2019-11-30'
    GROUP BY d.date, d.day_name, d.is_weekend, d.is_black_friday
)
SELECT
    date,
    day_name,
    sessions,
    revenue,
    CASE 
        WHEN COUNT(*) OVER (ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) = 7 
        THEN ROUND(AVG(sessions) OVER (ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 0)
    END AS sessions_7d_avg,
    CASE 
        WHEN COUNT(*) OVER (ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) = 7 
        THEN ROUND(AVG(revenue) OVER (ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2)
    END AS revenue_7d_avg
FROM daily_rollup
ORDER BY date ASC;

In [0]:
WITH weekly_rollup AS (
    SELECT
        d.week_start_date,
        COUNT(DISTINCT d.date)                                                AS days,
        COUNT(s.user_session)                                                 AS sessions,
        ROUND(SUM(COALESCE(s.revenue, 0)), 2)                                 AS revenue,
        ROUND(SUM(COALESCE(s.revenue, 0)) / NULLIF(COUNT(s.user_session), 0), 2) AS revenue_per_session
    FROM dbw_kroger_insights.gold.dim_date d
    LEFT JOIN dbw_kroger_insights.gold.ec_fact_sessions s
        ON s.session_date = d.date
    WHERE d.date BETWEEN DATE'2019-09-30' AND DATE'2019-12-01'
    GROUP BY d.week_start_date
),
weekly_lag AS (
    SELECT
        week_start_date,
        days,
        sessions,
        revenue,
        revenue_per_session,
        LAG(revenue) OVER (ORDER BY week_start_date)                          AS prev_week_revenue
    FROM weekly_rollup
)
SELECT
    week_start_date,
    days,
    sessions,
    revenue,
    revenue_per_session,
    prev_week_revenue,
    ROUND(
        (revenue - prev_week_revenue) * 100.0 / NULLIF(prev_week_revenue, 0), 2
    )                                                                         AS revenue_wow_pct
FROM weekly_lag
ORDER BY week_start_date ASC;

In [0]:
CREATE OR REPLACE TABLE gold.kpi_daily
COMMENT 'Daily eCommerce KPIs for dashboards. One row per calendar day (Oct–Nov 2019). 7-day averages are null until the window is full. is_anomaly_window flags Nov 14–17 (sale event + Nov 15 checkout/tracking incident).'
AS
WITH daily AS (
    SELECT
        d.date,
        d.date_key,
        d.day_name,
        d.week_start_date,
        d.year_month,
        d.is_weekend,
        d.is_black_friday,
        d.date BETWEEN DATE'2019-11-14' AND DATE'2019-11-17'                     AS is_anomaly_window,
        COUNT(s.user_session)                                                     AS sessions,
        COUNT_IF(s.has_purchase)                                                  AS converting_sessions,
        ROUND(COALESCE(SUM(s.revenue), 0), 2)                                     AS revenue
    FROM gold.dim_date d
    LEFT JOIN gold.ec_fact_sessions s
        ON s.session_date = d.date
    GROUP BY d.date, d.date_key, d.day_name, d.week_start_date, d.year_month,
             d.is_weekend, d.is_black_friday
)
SELECT
    *,
    ROUND(100.0 * converting_sessions / NULLIF(sessions, 0), 2)                   AS session_conversion_pct,
    ROUND(revenue / NULLIF(sessions, 0), 2)                                        AS revenue_per_session,
    CASE WHEN COUNT(*) OVER w = 7 THEN ROUND(AVG(sessions) OVER w, 0) END          AS sessions_7d_avg,
    CASE WHEN COUNT(*) OVER w = 7 THEN ROUND(AVG(revenue)  OVER w, 2) END          AS revenue_7d_avg
FROM daily
WINDOW w AS (ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW);

In [0]:
SELECT COUNT(*) AS days, SUM(sessions) AS sessions, ROUND(SUM(revenue), 2) AS revenue,
       COUNT_IF(is_anomaly_window) AS anomaly_days
FROM gold.kpi_daily;